In [14]:
# Setup once
from pathlib import Path
from scipy.signal import butter, filtfilt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

folder_path = Path("/Users/xylu/Desktop/Data/electrode")
csv_files = sorted(folder_path.glob("*.csv"))

# Pre-group files by RFD prefix
# File format: YYYYMMDD_HHMMSS_rfdXtbsY.csv
# rfdX: D0X Electrode (e.g., rfd4 = D04 Electrode, rfd10 = D10 Electrode, rfd11 = D11 Electrode)

files_by_rfd = {}
for file in csv_files:
    rfd = file.stem.split('_')[-1]
    if rfd not in files_by_rfd:
        files_by_rfd[rfd] = []
    files_by_rfd[rfd].append(file)

def plot_rfd(rfd_prefix: str, files_dict, xlim=None, ylims=None, channels=None, enable_filter=False, cutoff_freq=None, filter_order=4):
    if channels is None:
        channels = [1, 2, 3]
    # Ensure CH4 is not processed even if passed
    channels = [ch for ch in channels if ch in (1, 2, 3)]
    
    matched = files_dict.get(rfd_prefix, [])
    if not matched:
        print(f"No files found for {rfd_prefix}")
        return

    n_files = len(matched)
    fig, axes = plt.subplots(n_files, 1, figsize=(14, 4 * n_files))
    if n_files == 1:
        axes = [axes]

    # Pre-compute y-limits if needed
    y_min, y_max = None, None
    if ylims:
        lows = [v[0] for v in ylims.values() if v and v[0] is not None]
        highs = [v[1] for v in ylims.values() if v and v[1] is not None]
        y_min = min(lows) if lows else None
        y_max = max(highs) if highs else None

    # Map channels to column names and read only needed columns (CH4 omitted)
    col_map = {1: 'CH1', 2: 'CH2', 3: 'CH3'}
    usecols = [col_map[ch] for ch in channels if ch in col_map]
    color_map = {1: '#d62728', 2: 'blue', 3: 'green'}

    for idx, file in enumerate(matched):
        df = pd.read_csv(file, skiprows=4, usecols=usecols)
        num_points = df.shape[0]
        time_us = np.arange(num_points) * 75e-6
        
        # Apply low-pass filter if requested (only to channels being plotted)
        if enable_filter and cutoff_freq:
            sampling_rate = 1 / (75e-9)  # Hz
            nyquist = sampling_rate / 2
            normalized_cutoff = cutoff_freq / nyquist
            if normalized_cutoff < 1:
                b, a = butter(filter_order, normalized_cutoff, btype='low')
                for ch in channels:
                    ch_name = col_map[ch]
                    if ch_name in df.columns:
                        df[ch_name] = filtfilt(b, a, df[ch_name])
        
        # Parse filename for electrode and test type info
        stem = file.stem
        parts = stem.split('_')[-1]  # e.g., "rfd4tbs2"
        rfd_part = parts.split('tbs')[0]  # e.g., "rfd4"
        electrode_num = rfd_part[3:]  # e.g., "4" or "10"
        electrode_str = f"D{int(electrode_num):02d}"  # e.g., "D04" or "D10"
        test_type = parts.split('tbs')[1]  # e.g., "2"
        test_type_map = {'1': 'LER abort', '2': 'rising edge', '3': 'falling edge'}
        if electrode_num == '11':
            d11_map = {'1': 'Exp.', '2': 'Arc'}
            label_suffix = d11_map.get(test_type, test_type_map.get(test_type, test_type))
            title_label = f"D11 Electrode ({label_suffix})"
        else:
            label_suffix = test_type_map.get(test_type, test_type)
            title_label = f"{electrode_str} Electrode ({label_suffix})"
        
        ax = axes[idx]
        for ch in channels:
            ch_name = col_map[ch]
            if ch_name in df.columns:
                color = color_map.get(ch)
                ax.plot(time_us, df[ch_name], label=ch_name, linewidth=1, color=color, marker='o', markersize=4, alpha=0.6)

        ax.set_xlabel('Time (µs)', fontsize=16)
        ax.set_ylabel('Voltage (V)', fontsize=16)
        ax.set_title(f'{title_label} | {file.name}', fontsize=14, fontweight='bold')
        ax.legend(fontsize=16, loc='upper right')
        ax.grid(True, alpha=0.3)
        ax.tick_params(labelsize=14)
        ax.set_xlim(left=xlim[0] if xlim else 0, right=xlim[1] if xlim else None)
        if y_min is not None or y_max is not None:
            ax.set_ylim(bottom=y_min, top=y_max)

    plt.tight_layout()
    plt.show()

In [15]:
# Batch plot: rfd4 (CH1 only)
xlim_rfd4 = None  # e.g., (0, 1500)
ylims_rfd4 = {}   # e.g., {"CH1": (0, 5), "CH2": (-1, 2)}
plot_rfd("rfd4", files_by_rfd, xlim=xlim_rfd4, ylims=ylims_rfd4, channels=[1])

No files found for rfd4


In [18]:
# Batch plot: rfd10 (CH1 only)
xlim_rfd10 = None  # e.g., (0, 1500)
ylims_rfd10 = {}   # e.g., {"CH1": (0, 5)}
channels_rfd10 = [1]

# Report files where CH1 peak exceeds 3 V
rfd10_files = files_by_rfd.get("rfd10", [])
over3v_d10 = []
for f in rfd10_files:
    df_tmp = pd.read_csv(f, skiprows=4, usecols=["CH1"])
    if df_tmp["CH1"].max() > 3.0:
        over3v_d10.append(f.name)
if over3v_d10:
    print("D10 CH1 > 3 V:")
    for name in over3v_d10:
        print(name)
else:
    print("D10 CH1 > 3 V: none")

plot_rfd("rfd10", files_by_rfd, xlim=xlim_rfd10, ylims=ylims_rfd10, channels=channels_rfd10)

D10 CH1 > 3 V: none
No files found for rfd10


In [ ]:
# Batch plot: rfd11 (use CH1-3)
xlim_rfd11 = None  # e.g., (0, 1500)
ylims_rfd11 = {}   # e.g., {"CH1": (0, 5)}
channels_rfd11 = [1, 2, 3]

# Report files where CH3 peak exceeds 2 V
rfd11_files = files_by_rfd.get("rfd11tbs1", []) + files_by_rfd.get("rfd11tbs2", []) + files_by_rfd.get("rfd11tbs3", [])
over2v_d11ch3 = []
for f in rfd11_files:
    df_tmp = pd.read_csv(f, skiprows=4, usecols=["CH3"]) if Path(f).exists() else None
    if df_tmp is not None and df_tmp["CH3"].max() > 2.0:
        over2v_d11ch3.append(f.name)
if over2v_d11ch3:
    print("D11 CH3 > 2 V:")
    for name in over2v_d11ch3:
        print(name)
else:
    print("D11 CH3 > 2 V: none")

# Plot each rfd11 variant separately
for rfd_key in ["rfd11tbs1", "rfd11tbs2", "rfd11tbs3"]:
    if rfd_key in files_by_rfd:
        plot_rfd(rfd_key, files_by_rfd, xlim=xlim_rfd11, ylims=ylims_rfd11, channels=channels_rfd11)

No files found for rdf11
No files found for rdf11
No files found for rdf11
No files found for rdf11
No files found for rdf11
No files found for rdf11
No files found for rdf11
No files found for rdf11
No files found for rdf11
